## 0.1 Prepare dataset

In [ ]:
import requests
import tarfile
from pathlib import Path
import shutil
import tempfile

# Dataset URL
dataset_url = "http://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz"

# Use path for temporary files, then copy to data directory
tmp_dir = Path(tempfile.gettempdir())
dataset_file = tmp_dir / "booksummaries.tar.gz"

# Path to data directory
data_dir = Path.cwd()
target_file = data_dir / "booksummaries.txt"

if not target_file.exists():
    print(f"Downloading CMU Book Summary Dataset from {dataset_url}...")

    # Download the file to /tmp
    response = requests.get(dataset_url, stream=True)
    response.raise_for_status()

    dataset_file.write_bytes(response.content)
    print(f"Downloaded to {dataset_file}")

    # Extract the archive
    print("Extracting archive...")
    with tarfile.open(dataset_file, "r:gz") as tar:
        tar.extractall(path=tmp_dir, filter='data')

    # Copy file to data directory
    extracted_file = tmp_dir / "booksummaries" / "booksummaries.txt"
    shutil.copy(str(extracted_file), str(target_file))

    # Cleanup
    shutil.rmtree(tmp_dir / "booksummaries")
    dataset_file.unlink()

    print(f"✓ Dataset ready: {target_file}")
else:
    print(f"✓ Dataset already exists: {target_file}")

## 1. Imports

In [ ]:
!pip install bertopic
!pip install bertopic[visualization]
!pip install pandas scikit-learn nltk
!pip install sentence-transformers umap-learn plotly

In [1]:
import pandas as pd
from bertopic import BERTopic
import nltk
from sklearn.feature_extraction.text import CountVectorizer
import os
import numpy as np

nltk.download('stopwords')
nltk.download('punkt') # for tokenization
nltk.download('wordnet') # contains word lemmas
nltk.download('omw-1.4') # for lemmatization
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

C:\Programming\book-topic-graph\data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mrsha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mrsha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mrsha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mrsha\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mrsha\AppData\Roaming\nltk_data...
[nltk

## 2 Loading and preliminary data cleaning

We load the dataset and remove rows that do not contain plot summaries. We also drop duplicate book titles to ensure each book is unique.

In [2]:
file_path = 'booksummaries.txt'

if not os.path.exists(file_path):
    print(f"File {file_path} not foud.")
else:
    column_names = ['wiki_id', 'freebase_id', 'book_title', 'author', 'publication_date', 'genres', 'plot_summary']
    df = pd.read_csv(file_path, sep='\t', header=None, names=column_names)

    df.dropna(subset=['plot_summary'], inplace=True)
    df.drop_duplicates(subset=['book_title'], inplace=True)

    print(f"Loaded and processed {len(df)} books.")
    print("Sample Data:")
    display(df.head())

Loaded and processed 16277 books.
Sample Data:


,wiki_id,freebase_id,book_title,author,publication_date,genres,plot_summary
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"{""/m/016lj8"": ""Roman \u00e0 clef"", ""/m/06nbt"":...","Old Major, the old boar on the Manor Farm, ca..."
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962,"{""/m/06n90"": ""Science Fiction"", ""/m/0l67h"": ""N...","Alex, a teenager living in near-future Englan..."
2,986,/m/0ldx,The Plague,Albert Camus,1947,"{""/m/02m4t"": ""Existentialism"", ""/m/02xlf"": ""Fi...",The text of The Plague is divided into five p...
3,1756,/m/0sww,An Enquiry Concerning Human Understanding,David Hume,NaN,NaN,The argument of the Enquiry proceeds by a ser...
4,2080,/m/0wkt,A Fire Upon the Deep,Vernor Vinge,NaN,"{""/m/03lrw"": ""Hard science fiction"", ""/m/06n90...",The novel posits that space around the Milky ...


### 2.1 More thorough data cleaning

In [3]:
import re
import html

def clean_summary_text(text):
    """
    Cleans the plot summary text by removing HTML tags, wiki markup, and other unwanted patterns.
    """
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = re.sub(r'==+.*?==+', ' ', text)
    text = re.sub(r'~Plot outline description~', '', text, flags=re.IGNORECASE)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\{\{.*?\}\}', ' ', text, flags=re.DOTALL)
    text = re.sub(r'\[\[(?:[^\|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


In [4]:
df['plot_summary'] = df['plot_summary'].apply(clean_summary_text)

num_books_before = len(df)
print(f"Number of books before cleaning: {num_books_before}")

MIN_LENGTH_THRESHOLD = 50

is_valid_summary = df['plot_summary'].str.len() >= MIN_LENGTH_THRESHOLD
removed_books = df[~is_valid_summary]
df = df[is_valid_summary].reset_index(drop=True)
num_books_after = len(df)
print(f"Number of books after cleaning: {num_books_after} (removed {num_books_before - num_books_after} books with short summaries)")

if not removed_books.empty:
    print("\nExamples of removed books due to short summaries:")
    display(removed_books[['book_title', 'plot_summary']].head(10))

Number of books before cleaning: 16277
Number of books after cleaning: 16220 (removed 57 books with short summaries)

Examples of removed books due to short summaries:


,book_title,plot_summary
2045,The Kennel Murder Case,~Plot outline description
3071,The Ring of Charon,- --> it:L'anello di Caronte
3879,Slavers,==Publication histor
4045,Deathstalker,- --> <!-
5271,Golem in the Gears,pl:Zakochany golem
5595,The Adventures of Super Diaper Baby,
5693,The Deathlord of Ixia,==Receptio
5879,The Caverns of Kalte,==Receptio
5972,The Eyes of Darkness,==Character
6335,Created By,~Plot outline description


## 3. Different wizualization methods

### 3.1 Transformers-based Embeddings

##### 3.1.1 Generating Embeddings with Sentence Transformers

In [5]:
from sentence_transformers import SentenceTransformer

if 'df' in locals():
    model = SentenceTransformer('all-MiniLM-L6-v2')
    summaries = df['plot_summary'].tolist()

    print(f"Starting to generate embeddings for {len(summaries)} books...")
    embeddings = model.encode(summaries, show_progress_bar=True)
    print(f"Created embeddings with shape: {embeddings.shape}")

Starting to generate embeddings for 16220 books...


Batches: 100%|██████████| 507/507 [00:12<00:00, 39.36it/s] 


Created embeddings with shape: (16220, 384)


#### 3.1.2 Saving the embeddings

In [6]:
embeddings_file = 'book_embeddings.npy'

print("Saving embeddings to file...")
np.save(embeddings_file, embeddings)

print(f"Embeddings saved to {embeddings_file}. Data shape: {embeddings.shape}")

Saving embeddings to file...
Embeddings saved to book_embeddings.npy. Data shape: (16220, 384)


##### 3.1.3 Dimensionality Reduction with UMAP

In [7]:
# Optional embeddings loading
embeddings_file = 'book_embeddings.npy'

print("Loading embeddings from file...")
embeddings = np.load(embeddings_file)
print(f"Loaded embeddings with shape: {embeddings.shape}")

Loading embeddings from file...
Loaded embeddings with shape: (16220, 384)


In [8]:
import umap

if 'embeddings' in locals():
    reducer = umap.UMAP(
        n_neighbors=15,
        n_components=2,     # 2d
        min_dist=0.1,
        metric='cosine',
        random_state=42
    )

    print("Starting dimensionality reduction (UMAP)...")
    umap_embeddings = reducer.fit_transform(embeddings)
    print(f"New shape after reduction: {umap_embeddings.shape}")

Starting dimensionality reduction (UMAP)...
New shape after reduction: (16220, 2)


In [9]:
#optional reduced embeddings saving
umap_embeddings_file = 'umap_embeddings.npy'

print("Saving embeddings 2d to file...")
np.save(umap_embeddings_file, umap_embeddings)

print(f"Embeddings saved to {umap_embeddings_file}. Data shape: {umap_embeddings.shape}")

Saving embeddings 2d to file...
Embeddings saved to umap_embeddings.npy. Data shape: (16220, 2)


##### 3.1.4 Visualization with Plotly

In [10]:
import plotly.express as px

if 'df' in locals():
    df_transformers = df.copy()

    df_transformers['x'] = umap_embeddings[:, 0]
    df_transformers['y'] = umap_embeddings[:, 1]

    print("Generating interactive plot...")

    fig = px.scatter(
        df_transformers,
        x='x',
        y='y',
        hover_data=['book_title', 'author'],
        title="Book similarity map",
        template='plotly_dark'
    )

    fig.update_traces(marker=dict(size=4, opacity=0.7))
    fig.update_layout(legend_title_text='Main Genre')

    fig.show()

Generating interactive plot...


#### 3.1.5 Adding a new book

In [11]:
new_book = {
    'title': "Martyr!",
    'author': "Kaveh Akbar",
    'plot_summary': """
Cyrus is a queer poet living in Indiana, recovering from addiction to alcohol and drugs. His father, now deceased, was an Iranian migrant worker on a farm in rural Indiana. Cyrus believes that when he was a baby, his mother was killed on Iran Air Flight 655, a passenger plane which was shot down by a US missile during the Iran-Iraq war.

Cyrus is interested in the idea of martyrdom, and begins working on a "book of martyrs", while considering his own conceptual suicide as a potential martyr. He hears of an Iranian performance artist named Orkideh, who has terminal breast cancer, and is spending her last days in the Brooklyn Museum as part of a Marina Abramović-esque performance piece named "Death-Speak". Cyrus travels to New York to talk with Orkideh, bringing his roommate Zee. Zee has strong feelings for Cyrus, although their relationship is often fraught.

The book cycles between time periods and perspectives, including chapters told by Cyrus' mother, who was in a secret lesbian relationship in Iran, and his uncle, who, while serving in the Iran-Iraq war, was instructed to dress as an "angel" on horseback in order to comfort dying Iranian soldiers on the battlefield.

Upon Orkideh's death, Cyrus finds out from Sang - Orkideh's gallerist and ex-wife - that Orkideh was actually his mother, Roya. Roya had swapped passports with her lover Leila in order to escape Iran, and Leila was killed on the plane. In a dreamlike final scene, Cyrus appears to reconcile with Zee, before walking into a pool of golden light.
    """
}

new_embedding = model.encode([new_book['plot_summary']])

new_point = reducer.transform(new_embedding)

fig.add_scatter(
    x=[new_point[0, 0]],
    y=[new_point[0, 1]],
    mode="markers+text",
    text=["New Book"],
    marker=dict(size=12, color="red", symbol="x"),
    name="New Book"
)
fig.show()

### 3.2 Classic Method

### 3.3 BERTopic

#### 3.3.1 Text Preprocessing

We create a function to clean the plot summaries: we remove special characters, stop-words (commonly occurring words like 'the' and 'a') and perform lemmatization (reducing words to their base form).

In [12]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

df_topic = df.copy()

def preprocess_text(text):
    text = text.lower()

    # Removing special characters and digits
    text = re.sub(r'\W|\d', ' ', text)

    tokens = nltk.word_tokenize(text)

    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(tokens)

if 'df' in locals():
    print("Beginning text processing...")
    df_topic['processed_summary'] = df_topic['plot_summary'].apply(preprocess_text)
    print("Text processing finished.")

    print("\n--- Example --- ")
    print("Original summary:")
    print(df_topic.iloc[0]['plot_summary'][:500])
    print("\nSummary after processing:")
    print(df_topic.iloc[0]['processed_summary'][:500])

Beginning text processing...
Text processing finished.

--- Example --- 
Original summary:
Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the animals a revolutionary song, 'Beasts of England'. When Major dies, two young pigs, Snowball and Napoleon, assume command and turn his dream into a philosophy. The animals revolt and drive the drunken and irresponsible Mr Jones from the farm, renaming it "Animal Farm". They adopt Seven Commandments of Animal-ism, the most important of which is, "All an

Summary after processing:
old major old boar manor farm call animal farm meeting compare human parasite teach animal revolutionary song beast england major dy two young pig snowball napoleon assume command turn dream philosophy animal revolt drive drunken irresponsible jones farm renaming animal farm adopt seven commandment animal ism important animal equal snowball attempt teach animal reading writing f

#### 3.3.2 Training BERTopic

BERTopic will automatically extract topics from the processed summaries. `CountVectorizer` is used to remove English stop-words during the creation of the topic representation.

In [13]:
if 'df_topic' in locals():
    documents = df_topic['processed_summary'].tolist()

    vectorizer_model = CountVectorizer(stop_words="english")

    # min_topic_size - help to avoid very small topics
    topic_model = BERTopic(
        language="english",
        vectorizer_model=vectorizer_model,
        min_topic_size=20,
        verbose=True
    )

    topics, probabilities = topic_model.fit_transform(documents)

    print("\nModel has been trained successfully!")
    print(f"Found {len(topic_model.get_topic_info())} topics.")

2025-12-04 15:09:52,177 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 507/507 [00:10<00:00, 49.40it/s] 
2025-12-04 15:10:04,989 - BERTopic - Embedding - Completed ✓
2025-12-04 15:10:04,990 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-04 15:10:08,454 - BERTopic - Dimensionality - Completed ✓
2025-12-04 15:10:08,456 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-04 15:10:09,051 - BERTopic - Cluster - Completed ✓
2025-12-04 15:10:09,057 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-04 15:10:11,135 - BERTopic - Representation - Completed ✓



Model has been trained successfully!
Found 63 topics.


#### 3.3.3 Visualization and results analysis

**Most frequent topics**

In [14]:
if 'topic_model' in locals():
    display(topic_model.get_topic_info())

,Topic,Count,Name,Representation,Representative_Docs
0,-1,9915,-1_life_time_book_story,"[life, time, book, story, year, new, man, fath...",[gospel according larry revolves around sevent...
1,0,738,0_murder_case_killer_detective,"[murder, case, killer, detective, crime, polic...",[local crime buff meet monthly small town lawr...
2,1,532,1_king_dragon_magic_power,"[king, dragon, magic, power, lord, imriel, cit...",[story set day conclusion taran wanderer nearl...
3,2,522,2_earth_human_planet_ship,"[earth, human, planet, ship, space, alien, cre...",[story begin year year future time novel writi...
4,3,322,3_novel_story_life_book,"[novel, story, life, book, character, year, re...",[protagonist born mid father premature death s...
...,...,...,...,...,...
58,57,22,57_climate_energy_carbon_warming,"[climate, energy, carbon, warming, global, emi...",[book consists three part epilogue drawing fre...
59,58,21,58_rayford_believer_tribulation_carpathia,"[rayford, believer, tribulation, carpathia, ni...",[cameron buck williams rayford steele become i...
60,59,21,59_ook_gluk_goppernopper_mog,"[ook, gluk, goppernopper, mog, goppernoppers, ...",[every night water strangely disappears new ta...
61,60,21,60_tom_swift_airship_invention,"[tom, swift, airship, invention, bumper, damon...",[story open discussion barton swift old friend...


**Visualization of topics as a bar chart**

This chart shows the 10 most popular topics and the keywords that define them.

In [15]:
if 'topic_model' in locals():
    display(topic_model.visualize_barchart(top_n_topics=10))

**Map of distances between topics**

It shows topics as bubbles. The size of a bubble corresponds to its popularity. Bubbles located close to each other are thematically similar. It help to understand the relationships between topics.

In [15]:
if 'topic_model' in locals():
    display(topic_model.visualize_topics())

**Hierarchical clustering of topics**

Shows how topics can be grouped into larger clusters. Useful for identifying overarching themes in literature.

In [16]:
if 'topic_model' in locals():
    display(topic_model.visualize_hierarchy())

**Saving the Model**

We save the trained model for easy reuse later. We also save the processed data for use in an API.

In [17]:
if 'topic_model' in locals():
    model_dir = 'model'
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    model_path = os.path.join(model_dir, 'cmu_books_bertopic_model')
    topic_model.save(model_path, serialization='safetensors')

    df_topic.to_csv('processed_book_data.csv', index=False)

    print(f"Model has been saved in: {model_path}")
    print("Precessed data has been saved in: processed_book_data.csv")

Model has been saved in: model\cmu_books_bertopic_model
Precessed data has been saved in: processed_book_data.csv
